<a href="https://colab.research.google.com/github/Aniketh78/Generative-AI-Lab_Experiments/blob/main/genAiExp08.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:


from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments
from datasets import Dataset
import evaluate
import torch

data = {
    "question": [
        "How to reset password?",
        "How to check balance?",
        "How to contact support?"
    ],
    "answer": [
        "Click on forgot password and follow instructions.",
        "Login to your account dashboard to view balance.",
        "Contact support via email or phone."
    ]
}

dataset = Dataset.from_dict(data)

def preprocess(example):
    return {
        "input_text": "Question: " + example["question"],
        "target_text": example["answer"]
    }

dataset = dataset.map(preprocess)

tokenizer = T5Tokenizer.from_pretrained("t5-small")
model = T5ForConditionalGeneration.from_pretrained("t5-small")

def tokenize(example):
    inputs = tokenizer(example["input_text"], padding="max_length", truncation=True)
    outputs = tokenizer(example["target_text"], padding="max_length", truncation=True)
    inputs["labels"] = outputs["input_ids"]
    return inputs

tokenized_dataset = dataset.map(tokenize)

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=2
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

trainer.train()

model.save_pretrained("model")
tokenizer.save_pretrained("model")

bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")

predictions = ["Click on forgot password and follow instructions."]
references = [["Click on forgot password and follow instructions."]]

print("BLEU:", bleu.compute(predictions=predictions, references=references))
print("ROUGE:", rouge.compute(predictions=predictions, references=["Click on forgot password and follow instructions."]))

input_text = "Question: How to reset password?"
input_ids = tokenizer.encode(input_text, return_tensors="pt")
output = model.generate(input_ids)
print(tokenizer.decode(output[0], skip_special_tokens=True))

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

BLEU: {'bleu': 1.0, 'precisions': [1.0, 1.0, 1.0, 1.0], 'brevity_penalty': 1.0, 'length_ratio': 1.0, 'translation_length': 8, 'reference_length': 8}
ROUGE: {'rouge1': np.float64(1.0), 'rouge2': np.float64(1.0), 'rougeL': np.float64(1.0), 'rougeLsum': np.float64(1.0)}
True
